# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors
This notebook demonstrates how to explore and process the FAIR² colorectal cancer dataset using the `mlcroissant` library. The dataset is described with a [Croissant schema](https://mlcommons.org/croissant/) and is suitable for programmatic FAIR data workflows.

### Dataset Source
* [FAIR² Croissant schema (JSON-LD)](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Use `mlcroissant` to load the dataset metadata and preview its description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields. All entities are referenced by their `@id`.

In [ ]:
# List record sets and their @id's
record_sets = dataset.record_sets
print(f"Record sets found: {len(record_sets)} record set(s)")
for recset in record_sets:
    print(f"- Record Set Name: {recset.name}, @id={recset.id}")
    if hasattr(recset, 'fields') and recset.fields:
        print(f"  Fields:")
        for field in recset.fields:
            print(f"    · {field.name} (id: {field.id}, type: {getattr(field, 'data_type', '-')})")
    if hasattr(recset, 'columns') and recset.columns:
        print(f"  Columns:")
        for col in recset.columns:
            print(f"    · {col.name} (id: {col.id}, type: {getattr(col, 'data_type', '-')})")

## 3. Data Extraction
Load each record set into a Pandas DataFrame for further analysis. The record sets and fields are referenced by their `@id`.


In [ ]:
# Extract all record sets into DataFrames, using @id
dataframes = {}
all_recordset_ids = [recset.id for recset in dataset.record_sets]

for rs_id in all_recordset_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display structure for the (first) main record set
if all_recordset_ids:
    main_rs_id = all_recordset_ids[0]
    print(f"Columns in record set {main_rs_id}:\n", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data transformations such as filtering, normalization, and grouping. All field and record set references use their `@id`.


In [ ]:
# Example: Filter and normalize a numeric field, group by a categorical field
import numpy as np

if all_recordset_ids:
    df = dataframes[main_rs_id]
    print(f"Preview first few records:\n{df.head()}\n")

    # Identify a numeric field by heuristic (int or float), else pick one
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for EDA: {numeric_field_id}")
    else:
        # Fallback to a plausible field name if types not available
        plausible_numeric = [col for col in df.columns if "age" in col.lower() or "interval" in col.lower() or "size" in col.lower()]
        numeric_field_id = plausible_numeric[0] if plausible_numeric else None
    
    # Similarly select a categorical/group field
    group_candidates = [col for col in df.columns if df[col].dtype == object]
    group_field_id = None
    for col in group_candidates:
        if "sex" in col.lower() or "msi" in col.lower() or "site" in col.lower() or "location" in col.lower() or "histotype" in col.lower():
            group_field_id = col
            break
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].quantile(0.5)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        mean_ = filtered_df[numeric_field_id].mean()
        std_ = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / std_
        print(f"\nNormalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field detected for filtering and normalization.")
else:
    print("No data available to analyze.")

## 5. Visualization
Plot distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_recordset_ids and numeric_field_id and group_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Boxplot by group
    if group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Insufficient data or variable selection to generate visualizations.")

## 6. Conclusion

- We have loaded a FAIR² colorectal cancer dataset using the Croissant schema and `mlcroissant` library.
- Entities were accessed exclusively via their `@id` fields to maintain traceability and reproducibility.
- Exploratory steps demonstrated how to inspect, filter, and visualize clinical and molecular variables from the main record set.
- This workflow provides a reproducible and FAIR approach to biomedical data exploration and analytics.